# Workflow Debugging

When running complex computations (such as workflows) on complex computing infrastructure (for example HPC clusters), things will go wrong. It is therefore important to understand how to detect and debug issues as they appear. The good news is that Pegasus is doing a good job with the detection part, using for example exit codes, and provides tooling to help you debug. In this notebook, we will be using the same workflow as in the previous one, but introduce an error and see if we can detect it. 

To introduce the error, let's rename the input to something which will mismatch the workflow description:

In [ ]:
!mv inputs/f.in inputs/f.in.BADNAME

Now plan and run the workflow:

In [ ]:
from Pegasus.api import *
import sys
import time
from pathlib import Path

import logging

logging.basicConfig(level=logging.DEBUG)

# we specify directories for inputs, executables and outputs
# - directory where to pick up the inputs from a directory.
# - directory where the executables that the workflow uses are placed.
# - directory where the outputs should be placed.

BASE_DIR = Path(".").resolve()
INPUT_DIR = Path(BASE_DIR /  "inputs").resolve()
EXECUTABLES_DIR = Path(BASE_DIR / ".." /  "executables").resolve()
OUTPUT_DIR = Path(BASE_DIR /  "outputs").resolve() 

# the execution site where you job to run.
# local means the jobs run on ACCESS Pegasus itself.
# condorpool means jobs will run on a node provisioned from an ACCESS site such as jetstream
EXEC_SITE="local"

# --- Replicas -----------------------------------------------------------------
fin = File("f.in").add_metadata(creator="vahi")
rc = ReplicaCatalog()\
    .add_replica("local", fin, "{}/f.in".format(INPUT_DIR))\
    .write() # written to ./replicas.yml 

# --- Properties -----------------------------------------------------------------
props = Properties()
props["pegasus.mode"] = "tutorial" # speed up error short circuit
props.write()
 
# --- Workflow -----------------------------------------------------------------
wf = Workflow("hello-world")

fin = File("f.in")
finter = File("f.inter")
fout = File("f.out")

job_hello = Job("hello")\
                    .add_args("-T", "3", "-i", fin, "-o {}".format(finter))\
                    .add_inputs(fin)\
                    .add_outputs(finter, stage_out=False)

job_world = Job("world")\
                    .add_args("-T", "3", "-i", finter, "-o {}".format(fout))\
                    .add_inputs(finter)\
                    .add_outputs(fout)

wf.add_jobs(job_hello, job_world)

# --- Visualize the Workflow ---------------------------------------------------
try:
    wf.write()
    wf.graph(include_files=True, label="xform-id", output="graph.png")
except PegasusClientError as e:
    print(e)


## Run the Workflow

In [ ]:
try:
    wf.plan(input_dirs=[INPUT_DIR], sites=["condorpool"], transformations_dir=EXECUTABLES_DIR,\
            output_dir=OUTPUT_DIR, submit=True)\
      .wait()
except PegasusClientError as e:
    print(e)

Note the status bar and the state of the different jobs.

## 3. Analyze

When the workflow fails, we can use the Pegasus analyze tool to pinpoint the failure.

In [ ]:
try:
    wf.analyze()
except:
    pass

In the output we can see `ERROR:  Expected local file does not exist: /home/rynge/.../inputs/f.in'`, which means that the local file might not exist.

## Resolving the issue

The cause of the problem is a mismatch between the input file (`inputs/f.in.BADNAME`) and what we have specified in the workflow (`inputs/f.in`). The file in the input directory was misnamed to cause this issue for demonstration purposes.

Let's resolve the issue by renaming the wrongly named input file:

In [ ]:
mv inputs/f.in.BADNAME inputs/f.in

## Restart the workflow

We can now restart the workflow from where it stopped. Alternatively to the `run()`, you could `plan_submit()` a new instance, but in that case the workflow would start all the way from the beginning again.

In [ ]:
wf.run()
time.sleep(30)  # give the workflow some time to get started again
wf.wait()

## Statistics

Pegasus collects provenance information during the workflow execution. By default, Pegasus launches all jobs through a process called `kickstart`, which captures runtime provenance data for each job. This data includes details about the execution environment, input and output files, execution parameters, and performance metrics. 

The collected provenance information is stored in a relational database, allowing users to analyze and summarize workflow executions. Pegasus provides tools such as `pegasus-statistics` to facilitate this analysis. To get a high level summary of the data, run `workflow.statistics()`

In [ ]:
wf.statistics()